# Anime Stitch Pipeline — Results Viewer

This notebook loads the JSON produced by
`backend/benchmark/bench_anime_stitch.py` and visualises everything
that the benchmark generated — images, plots, metrics, intermediate
outputs — **without re-running any pipeline code**.

## Workflow
1. Run the benchmark: `python backend/benchmark/bench_anime_stitch.py`
2. Set `JSON_PATH` in the next cell to the generated JSON file
   (`backend/benchmark/results/anime_stitch_YYYYMMDD_HHMMSS.json`)
3. Run all cells (`Kernel → Restart & Run All`)

## 1. Configuration

In [ ]:
import glob
import json
import math
import os
from pathlib import Path
from typing import Dict, List, Optional

import cv2
import numpy as np
import matplotlib
matplotlib.rcParams.update({
    'figure.facecolor':  '#12121f',
    'axes.facecolor':    '#1a1a2e',
    'text.color':        'white',
    'axes.labelcolor':   'white',
    'xtick.color':       'white',
    'ytick.color':       'white',
    'axes.edgecolor':    '#555',
    'savefig.facecolor': '#12121f',
    'figure.titlesize':  12,
    'axes.titlesize':    10,
})
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import matplotlib.gridspec as gridspec
from IPython.display import display, Markdown

REPO = Path('/home/pkhunter/Repositories/Image-Toolkit')

# ── Set this to the JSON file you want to analyse ──────────────────────────
# Leave as None to auto-select the most recent file.
JSON_PATH: Optional[str] = None
# ───────────────────────────────────────────────────────────────────────────

if JSON_PATH is None:
    candidates = sorted(
        glob.glob(str(REPO / 'backend/benchmark/results/anime_stitch_*.json'))
    )
    if not candidates:
        raise FileNotFoundError(
            'No anime_stitch_*.json found in backend/benchmark/results/.\n'
            'Run: python backend/benchmark/bench_anime_stitch.py'
        )
    JSON_PATH = candidates[-1]

with open(JSON_PATH) as fh:
    DATA = json.load(fh)

DATASETS: List[Dict] = DATA['datasets']
SUMMARY  = DATA['summary']
SYSTEM   = DATA['system']
META     = DATA['metadata']
INSIGHTS = DATA['performance_insights']

print(f'Loaded: {JSON_PATH}')
print(f'Suite : {META["suite_name"]}')
print(f'Run at: {META["timestamp"]}')
print(f'Datasets: {META["total_datasets"]}  |  Total time: {META["total_time_sec"]}s')
print(f'GPU: {SYSTEM.get("gpu", "N/A")}  |  VRAM: {SYSTEM.get("vram_gb", "N/A")} GB')

## 2. System & Run Summary

In [ ]:
lines = [
    '| Field | Value |',
    '|-------|-------|',
]
for k, v in SYSTEM.items():
    lines.append(f'| `{k}` | {v} |')
display(Markdown('### System\n' + '\n'.join(lines)))

lines2 = [
    '| Metric | Value |',
    '|--------|-------|',
    f'| Total datasets | {SUMMARY["total_datasets"]} |',
    f'| Passed (full ASP) | {SUMMARY["datasets_passed"]} |',
    f'| SCANS fallback | {SUMMARY["datasets_fallback"]} |',
    f'| Total run time | {SUMMARY["total_time_sec"]}s |',
    f'| Avg time / dataset | {SUMMARY["avg_time_per_dataset_sec"]}s |',
    f'| Avg ASP sharpness | {SUMMARY.get("avg_sharpness_asp")} |',
    f'| Avg Simple sharpness | {SUMMARY.get("avg_sharpness_simple")} |',
    f'| Avg ASP ghosting | {SUMMARY.get("avg_ghosting_asp")} |',
    f'| Avg Simple ghosting | {SUMMARY.get("avg_ghosting_simple")} |',
    f'| Avg ASP coverage | {SUMMARY.get("avg_coverage_asp")} |',
    f'| Avg SSIM (ASP vs Simple) | {SUMMARY.get("avg_ssim")} |',
]
vc = SUMMARY.get('verdict_counts', {})
lines2 += [
    f'| ASP better | {vc.get("asp_better", 0)} datasets |',
    f'| Simple better | {vc.get("simple_better", 0)} datasets |',
    f'| Comparable | {vc.get("comparable", 0)} datasets |',
]
display(Markdown('### Run Summary\n' + '\n'.join(lines2)))

## 3. Global Metrics Dashboard

In [ ]:
names     = [d['name'] for d in DATASETS]
short     = [n.replace('asp_test', 'T') for n in names]

asp_sharp = [d['metrics_asp'].get('sharpness', 0)      for d in DATASETS]
sim_sharp = [d['metrics_simple'].get('sharpness', 0)   for d in DATASETS]
asp_ghost = [d['metrics_asp'].get('ghosting_score', 0) for d in DATASETS]
sim_ghost = [d['metrics_simple'].get('ghosting_score', 0) for d in DATASETS]
asp_cov   = [d['metrics_asp'].get('coverage', 0)       for d in DATASETS]
sim_cov   = [d['metrics_simple'].get('coverage', 0)    for d in DATASETS]
asp_seam  = [d['metrics_asp'].get('seam_gradient', 0)  for d in DATASETS]
sim_seam  = [d['metrics_simple'].get('seam_gradient', 0) for d in DATASETS]
ssims     = [d['comparison'].get('ssim') or 0          for d in DATASETS]
times     = [d['time'].get('total_sec', 0)              for d in DATASETS]
fallbacks = [d['used_fallback']                         for d in DATASETS]
verdicts  = [d['comparison'].get('verdict', '')         for d in DATASETS]

x  = np.arange(len(names))
w  = 0.38

fig, axes = plt.subplots(3, 2, figsize=(16, 14))

palette = {'asp': '#4ecdc4', 'simple': '#ff6b6b'}

def _bar_pair(ax, label_a, vals_a, label_b, vals_b, title, ylabel,
              invert_good=False):
    bars_a = ax.bar(x - w/2, vals_a, w, label=label_a,
                    color=palette['asp'], alpha=0.88)
    bars_b = ax.bar(x + w/2, vals_b, w, label=label_b,
                    color=palette['simple'], alpha=0.88)
    # Mark fallback datasets
    for xi, fb in zip(x, fallbacks):
        if fb:
            ax.axvline(xi, color='yellow', alpha=0.25, linewidth=6)
    ax.set_xticks(x)
    ax.set_xticklabels(short, fontsize=8)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend(facecolor='#2a2a3e', labelcolor='white', fontsize=8)
    ax.grid(axis='y', alpha=0.25)

_bar_pair(axes[0,0], 'ASP', asp_sharp, 'Simple', sim_sharp,
          'Sharpness (↑ better)', 'Laplacian variance')
_bar_pair(axes[0,1], 'ASP', asp_ghost, 'Simple', sim_ghost,
          'Ghosting Score (↓ better)', '2nd-order ∇')
_bar_pair(axes[1,0], 'ASP', asp_cov,   'Simple', sim_cov,
          'Coverage (↑ better)', 'non-black fraction')
_bar_pair(axes[1,1], 'ASP', asp_seam,  'Simple', sim_seam,
          'Seam Gradient (↓ better)', 'mean |∇y| at seams')

# SSIM
colors_ssim = ['#ff6b6b' if s < 0.5 else '#ffd93d' if s < 0.75 else '#4ecdc4'
               for s in ssims]
axes[2,0].bar(x, ssims, color=colors_ssim, alpha=0.88)
axes[2,0].set_xticks(x)
axes[2,0].set_xticklabels(short, fontsize=8)
axes[2,0].set_title('SSIM — ASP vs Simple (↑ = more similar)')
axes[2,0].set_ylabel('SSIM')
axes[2,0].set_ylim(0, 1)
axes[2,0].axhline(0.5, color='#ff6b6b', linestyle='--', alpha=0.5, label='0.5 threshold')
axes[2,0].legend(facecolor='#2a2a3e', labelcolor='white', fontsize=8)
axes[2,0].grid(axis='y', alpha=0.25)

# Processing time
fallback_colors = ['#ff6b6b' if fb else '#4ecdc4' for fb in fallbacks]
axes[2,1].bar(x, times, color=fallback_colors, alpha=0.88)
axes[2,1].set_xticks(x)
axes[2,1].set_xticklabels(short, fontsize=8)
axes[2,1].set_title('Processing Time per Dataset (red = SCANS fallback)')
axes[2,1].set_ylabel('Seconds')
axes[2,1].grid(axis='y', alpha=0.25)

fig.suptitle('Global Metrics Dashboard — All Datasets\n'
             '(yellow band = SCANS fallback used)',
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Verdict summary pie + scatter: sharpness vs ghosting
vc = SUMMARY.get('verdict_counts', {})
fig, (ax_pie, ax_scatter) = plt.subplots(1, 2, figsize=(12, 5))

pie_labels = [k for k, v in vc.items() if v > 0]
pie_vals   = [v for v in vc.values() if v > 0]
pie_colors = ['#4ecdc4', '#ff6b6b', '#ffd93d', '#a29bfe']
ax_pie.pie(pie_vals, labels=pie_labels, colors=pie_colors[:len(pie_vals)],
           autopct='%1.0f%%', textprops={'color': 'white'},
           wedgeprops={'linewidth': 1, 'edgecolor': '#12121f'})
ax_pie.set_title('Verdict Distribution (ASP vs Simple)')

verdict_color = {
    'asp_better': '#4ecdc4',
    'simple_better': '#ff6b6b',
    'comparable': '#ffd93d',
    'insufficient_data': '#aaa',
}
for d, xs, ys in zip(DATASETS, asp_sharp, asp_ghost):
    c = verdict_color.get(d['comparison'].get('verdict', ''), '#aaa')
    ax_scatter.scatter(xs, ys, color=c, s=60, alpha=0.85, zorder=3)
    ax_scatter.annotate(
        d['name'].replace('asp_test', 'T'),
        (xs, ys), textcoords='offset points', xytext=(4, 3),
        fontsize=7, color='white'
    )
for label, color in verdict_color.items():
    ax_scatter.scatter([], [], color=color, label=label, s=40)
ax_scatter.set_xlabel('ASP Sharpness')
ax_scatter.set_ylabel('ASP Ghosting Score (lower = better)')
ax_scatter.set_title('ASP Quality Space (sharpness vs ghosting)')
ax_scatter.legend(facecolor='#2a2a3e', labelcolor='white', fontsize=8)
ax_scatter.grid(alpha=0.2)

plt.suptitle('Verdict & Quality Space', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Timing Breakdown

In [ ]:
# Stacked bar chart: time breakdown per dataset
stage_keys = [
    ('simple_stitch_sec',  'Simple stitch',     '#636e72'),
    ('birefnet_sec',       'BiRefNet',           '#a29bfe'),
    ('matching_sec',       'Matching (LoFTR)',   '#fd79a8'),
    ('bundle_adjust_sec',  'Bundle adjust',      '#ffeaa7'),
    ('ecc_sec',            'ECC refinement',     '#55efc4'),
    ('render_sec',         'Temporal render',    '#4ecdc4'),
    ('composite_sec',      'FG composite',       '#74b9ff'),
    ('visualisations_sec', 'Visualisations',     '#dfe6e9'),
    ('scans_fallback_sec', 'SCANS fallback',     '#ff6b6b'),
]

fig, ax = plt.subplots(figsize=(16, 6))
bottom = np.zeros(len(DATASETS))
for key, label, color in stage_keys:
    vals = np.array([d['time'].get(key, 0.0) for d in DATASETS])
    if vals.sum() == 0:
        continue
    ax.bar(x, vals, bottom=bottom, label=label, color=color, alpha=0.88, width=0.6)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(short, fontsize=8)
ax.set_ylabel('Seconds')
ax.set_title('Stage Timing Breakdown per Dataset')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left',
          facecolor='#2a2a3e', labelcolor='white', fontsize=8)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

# Print table
header = f"{'Dataset':>14}  {'Total':>7}"
for _, label, _ in stage_keys:
    header += f"  {label[:10]:>10}"
print(header)
print('─' * len(header))
for d in DATASETS:
    t = d['time']
    row = f"{d['name']:>14}  {t.get('total_sec', 0):>7.1f}"
    for key, _, _ in stage_keys:
        row += f"  {t.get(key, 0):>10.2f}"
    print(row)

## 5. Alignment & Matching Analysis

In [ ]:
# Alignment health overview table
print(f"{'Dataset':>14}  {'Valid':>6}  {'Ratio':>7}  {'MinGap':>8}  "
      f"{'MaxRot':>8}  {'MaxScl':>8}  {'Reason'}")
print('─' * 80)
for d in DATASETS:
    ah = d['affine_health']
    ok = '✓' if ah['valid'] else '✗'
    print(
        f"{d['name']:>14}  {ok:>6}  {ah['ratio']:>7.3f}  "
        f"{ah['min_gap_px']:>8.1f}  {ah['max_rotation']:>8.5f}  "
        f"{ah['max_scale_dev']:>8.5f}  {ah['reason']}"
    )

In [ ]:
# Translation path visualisation — one subplot per dataset (small multiples)
ncols = 4
nrows = math.ceil(len(DATASETS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3))
axes_flat = axes.ravel() if hasattr(axes, 'ravel') else [axes]

for idx, (d, ax) in enumerate(zip(DATASETS, axes_flat)):
    align = d.get('alignment', {})
    affines = align.get('affines', [])
    if not affines:
        ax.set_title(d['name'].replace('asp_test', 'T'), fontsize=8)
        ax.axis('off')
        continue
    tys = [a['ty'] for a in affines]
    txs = [a['tx'] for a in affines]
    color = '#ff6b6b' if d['used_fallback'] or not d['affine_health']['valid'] else '#4ecdc4'
    ax.plot(tys, marker='o', color=color, linewidth=1.5, markersize=4, label='ty')
    ax.plot(txs, marker='s', color='#ffd93d', linewidth=1,  markersize=3, label='tx',
            linestyle='--', alpha=0.7)
    ax.set_title(d['name'].replace('asp_test', 'T'), fontsize=8,
                 color='#ff6b6b' if d['used_fallback'] else 'white')
    ax.set_xlabel('frame', fontsize=7)
    ax.set_ylabel('px', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(alpha=0.2)

for ax in axes_flat[len(DATASETS):]:
    ax.axis('off')

# Legend on first panel
axes_flat[0].legend(facecolor='#2a2a3e', labelcolor='white', fontsize=7)
fig.suptitle('Translation Vectors per Dataset  (red title = fallback/failed)',
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Δty coefficient of variation — uniformity of pan speed
dy_cvs  = [d['alignment'].get('dy_cv', 0) for d in DATASETS]
dx_cvs  = [d['alignment'].get('dx_cv', 0) for d in DATASETS]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
bar_colors = ['#ff6b6b' if v > 0.5 else '#4ecdc4' for v in dy_cvs]
axes[0].bar(x, dy_cvs, color=bar_colors, alpha=0.88)
axes[0].axhline(0.5, color='white', linestyle='--', alpha=0.5, label='0.5 threshold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(short, fontsize=8)
axes[0].set_title('Δty Coefficient of Variation  (> 0.5 = uneven pan)')
axes[0].set_ylabel('CV')
axes[0].legend(facecolor='#2a2a3e', labelcolor='white', fontsize=8)
axes[0].grid(axis='y', alpha=0.2)

bar_colors_x = ['#ff6b6b' if v > 0.5 else '#ffd93d' for v in dx_cvs]
axes[1].bar(x, dx_cvs, color=bar_colors_x, alpha=0.88)
axes[1].axhline(0.5, color='white', linestyle='--', alpha=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(short, fontsize=8)
axes[1].set_title('Δtx Coefficient of Variation  (> 0.5 = horizontal drift)')
axes[1].set_ylabel('CV')
axes[1].grid(axis='y', alpha=0.2)

plt.suptitle('Pan Uniformity — Translation Step Variability', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Matching method breakdown
all_methods = set()
for d in DATASETS:
    all_methods.update(d['matching'].get('methods', {}).keys())
all_methods = sorted(all_methods)

if all_methods:
    fig, (ax_methods, ax_edges) = plt.subplots(1, 2, figsize=(14, 4))
    method_colors = ['#a29bfe', '#fd79a8', '#ffeaa7', '#55efc4', '#4ecdc4']
    bottom = np.zeros(len(DATASETS))
    for method, color in zip(all_methods, method_colors):
        vals = np.array([
            d['matching'].get('methods', {}).get(method, 0) for d in DATASETS
        ])
        ax_methods.bar(x, vals, bottom=bottom, label=method, color=color, alpha=0.88)
        bottom += vals
    ax_methods.set_xticks(x)
    ax_methods.set_xticklabels(short, fontsize=8)
    ax_methods.set_title('Edges by Matching Method')
    ax_methods.set_ylabel('Edge count')
    ax_methods.legend(facecolor='#2a2a3e', labelcolor='white', fontsize=8)
    ax_methods.grid(axis='y', alpha=0.2)

    raw_edges      = [d['matching'].get('raw_edges', 0) for d in DATASETS]
    filtered_edges = [d['matching'].get('filtered_edges', 0) for d in DATASETS]
    ax_edges.bar(x - w/2, raw_edges, w, label='Raw',      color='#636e72', alpha=0.88)
    ax_edges.bar(x + w/2, filtered_edges, w, label='Filtered', color='#4ecdc4', alpha=0.88)
    ax_edges.set_xticks(x)
    ax_edges.set_xticklabels(short, fontsize=8)
    ax_edges.set_title('Raw vs Filtered Edge Count')
    ax_edges.set_ylabel('Edges')
    ax_edges.legend(facecolor='#2a2a3e', labelcolor='white', fontsize=8)
    ax_edges.grid(axis='y', alpha=0.2)

    plt.suptitle('Feature Matching Statistics', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('No method metadata available in this JSON.')

## 6. Photometric Correction Analysis

In [ ]:
# Gain range and frames corrected across all datasets
gain_mins    = [d['photometric'].get('gain_range', [1.0, 1.0])[0] for d in DATASETS]
gain_maxs    = [d['photometric'].get('gain_range', [1.0, 1.0])[1] for d in DATASETS]
frames_corr  = [d['photometric'].get('frames_corrected', 0) for d in DATASETS]
frame_counts = [d['frames'].get('count', 0) for d in DATASETS]
ref_lums     = [d['photometric'].get('ref_lum') or 0 for d in DATASETS]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Gain range plot
axes[0].bar(x, gain_maxs, color='#ff6b6b', alpha=0.7, label='max gain')
axes[0].bar(x, gain_mins, color='#4ecdc4', alpha=0.7, label='min gain')
axes[0].axhline(1.0, color='white', linestyle='--', alpha=0.5, label='1.0')
axes[0].axhline(1.14, color='#ffd93d', linestyle=':', alpha=0.5, label='clip max')
axes[0].axhline(0.88, color='#ffd93d', linestyle=':', alpha=0.5, label='clip min')
axes[0].set_xticks(x)
axes[0].set_xticklabels(short, fontsize=8)
axes[0].set_title('Luminance Gain Range')
axes[0].set_ylabel('Gain')
axes[0].legend(facecolor='#2a2a3e', labelcolor='white', fontsize=7)
axes[0].grid(axis='y', alpha=0.2)

# Frames corrected as a fraction
frac_corr = [c / max(n, 1) for c, n in zip(frames_corr, frame_counts)]
bar_colors_corr = ['#ff6b6b' if f > 0.5 else '#4ecdc4' for f in frac_corr]
axes[1].bar(x, frac_corr, color=bar_colors_corr, alpha=0.88)
axes[1].set_xticks(x)
axes[1].set_xticklabels(short, fontsize=8)
axes[1].set_title('Fraction of Frames Corrected  (> 50% = heavy exposure drift)')
axes[1].set_ylabel('Fraction')
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', alpha=0.2)

# Reference luminance
axes[2].bar(x, ref_lums, color='#a29bfe', alpha=0.88)
axes[2].set_xticks(x)
axes[2].set_xticklabels(short, fontsize=8)
axes[2].set_title('Reference Luminance (median BG lum)')
axes[2].set_ylabel('Luma (0–255)')
axes[2].grid(axis='y', alpha=0.2)

plt.suptitle('Photometric Correction Summary', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed per-frame gains for a selected dataset — reuse gains.png or replot from JSON
SEL_DATASET = 'asp_test1'   # ← change to inspect any dataset

sel = next((d for d in DATASETS if d['name'] == SEL_DATASET), None)
if sel is None:
    print(f'{SEL_DATASET} not found in results')
else:
    bg_lums = sel['photometric']['bg_lums']
    gains   = sel['photometric']['applied_gains']
    N_fr    = len(gains)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    lum_vals = [l if l is not None else 0 for l in bg_lums]
    axes[0].bar(range(N_fr), lum_vals, color='#4ecdc4', alpha=0.85)
    if sel['photometric']['ref_lum']:
        axes[0].axhline(sel['photometric']['ref_lum'], color='#ff6b6b',
                        linestyle='--', label=f'ref={sel["photometric"]["ref_lum"]:.1f}')
    axes[0].set_title(f'{SEL_DATASET} — BG Luminance per Frame')
    axes[0].set_xlabel('Frame')
    axes[0].set_ylabel('Luminance')
    axes[0].legend(facecolor='#2a2a3e', labelcolor='white', fontsize=8)
    axes[0].grid(axis='y', alpha=0.2)

    axes[1].bar(range(N_fr), gains, color='#ff6b6b', alpha=0.85)
    axes[1].axhline(1.0, color='#4ecdc4', linestyle='--', label='gain=1.0')
    axes[1].axhline(0.88, color='white', linestyle=':', alpha=0.4, label='clip [0.88, 1.14]')
    axes[1].axhline(1.14, color='white', linestyle=':', alpha=0.4)
    axes[1].set_title(f'{SEL_DATASET} — Applied Gains')
    axes[1].set_xlabel('Frame')
    axes[1].set_ylabel('Gain multiplier')
    axes[1].legend(facecolor='#2a2a3e', labelcolor='white', fontsize=8)
    axes[1].grid(axis='y', alpha=0.2)

    plt.tight_layout()
    plt.show()

## 7. Per-Dataset Saved Plots Viewer

Displays the plots that the benchmark already saved to disk —
no recomputation needed.

In [ ]:
def _load_bgr(path: str) -> Optional[np.ndarray]:
    img = cv2.imread(path) if path and os.path.exists(path) else None
    return img

def _show_plot(path: str, title: str, ax, fallback_text: str = 'not generated') -> None:
    img = _load_bgr(path)
    if img is not None:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), aspect='auto')
    else:
        ax.text(0.5, 0.5, fallback_text, ha='center', va='center',
                color='#888', transform=ax.transAxes, fontsize=9)
    ax.set_title(title, fontsize=8)
    ax.axis('off')

# Show one dataset at a time — change INSPECT_ID to switch
INSPECT_ID = 'asp_test1'   # ← change this

d = next((r for r in DATASETS if r['name'] == INSPECT_ID), None)
if d is None:
    print(f'{INSPECT_ID} not found.')
else:
    plots_dir = d['paths']['plots_dir']
    stage_dir = d['paths']['stage_dir']

    def p(fname): return os.path.join(plots_dir, fname)
    def s(fname): return os.path.join(stage_dir, fname)

    fig = plt.figure(figsize=(18, 28), constrained_layout=True)
    gs  = gridspec.GridSpec(7, 4, figure=fig)

    # Row 0: final outputs side by side
    ax_asp    = fig.add_subplot(gs[0, :2])
    ax_simple = fig.add_subplot(gs[0, 2:])
    _show_plot(d['paths']['anime_stitch'],  'ASP Final Output',           ax_asp)
    _show_plot(d['paths']['simple_stitch'], 'Simple Stitch Final Output', ax_simple)

    # Row 1: metrics bar + gains
    ax_metr  = fig.add_subplot(gs[1, :2])
    ax_gains = fig.add_subplot(gs[1, 2:])
    _show_plot(p('metrics_comparison.png'), 'CV Metrics Comparison', ax_metr)
    _show_plot(p('gains.png'),              'Per-Frame Gains',        ax_gains)

    # Row 2: canvas placement + overlap map
    ax_canvas  = fig.add_subplot(gs[2, :2])
    ax_overlap = fig.add_subplot(gs[2, 2:])
    _show_plot(p('canvas_frame_placement.png'), 'Canvas Frame Placement (2D)', ax_canvas)
    _show_plot(p('overlap_map.png'),             'Frame Overlap Count Map',    ax_overlap)

    # Row 3: translation vectors + seam heatmaps
    ax_trans    = fig.add_subplot(gs[3, :2])
    ax_seam_asp = fig.add_subplot(gs[3, 2:])
    _show_plot(p('translation_vectors.png'),  'Translation Vectors',       ax_trans)
    _show_plot(p('asp_seam_heatmap.png'),      'ASP Seam Heatmap (2D)',    ax_seam_asp)

    ax_seam_sim  = fig.add_subplot(gs[4, :2])
    ax_rend_3d   = fig.add_subplot(gs[4, 2:])
    _show_plot(p('simple_seam_heatmap.png'),   'Simple Seam Heatmap (2D)', ax_seam_sim)
    _show_plot(p('temporal_render_3d.png'),    'Stage 9 Render 3D Surface', ax_rend_3d)

    # Row 5: 3D surfaces
    ax_asp_3d = fig.add_subplot(gs[5, :2])
    ax_sim_3d = fig.add_subplot(gs[5, 2:])
    _show_plot(p('asp_3d_surface.png'),    'ASP Luminance 3D',    ax_asp_3d)
    _show_plot(p('simple_3d_surface.png'), 'Simple Luminance 3D', ax_sim_3d)

    # Row 6: stage images
    stage_files = [
        (s('stage09_temporal_render.png'), 'Stage 9 Render'),
        (s('stage11_fg_composite.png'),    'Stage 11 FG Composite'),
        (s('stage04_bgmask_frame00.png'),  'BG Mask F0'),
        (p('mask_overlay_frame00.png'),    'Mask Overlay F0'),
    ]
    for col, (fpath, title) in enumerate(stage_files):
        ax = fig.add_subplot(gs[6, col])
        _show_plot(fpath, title, ax)

    fig.suptitle(f'{INSPECT_ID} — Full Visual Report', fontsize=14)
    plt.show()

## 8. All-Dataset Side-by-Side Strip

A compact strip showing ASP vs Simple for every test at a glance.

In [ ]:
# Compact thumbnail grid: ASP (top row) vs Simple (bottom row)
THUMB_W, THUMB_H = 260, 380   # display thumbnail size

def _thumb(path: str) -> np.ndarray:
    img = cv2.imread(path) if path and os.path.exists(path) else None
    if img is None:
        ph = np.zeros((THUMB_H, THUMB_W, 3), dtype=np.uint8)
        cv2.putText(ph, 'N/A', (60, THUMB_H//2), cv2.FONT_HERSHEY_SIMPLEX,
                    1.2, (100, 100, 100), 2)
        return ph
    h, w = img.shape[:2]
    scale = min(THUMB_W / w, THUMB_H / h)
    nw, nh = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((THUMB_H, THUMB_W, 3), dtype=np.uint8)
    y0 = (THUMB_H - nh) // 2
    x0 = (THUMB_W - nw) // 2
    canvas[y0:y0+nh, x0:x0+nw] = resized
    return canvas

n = len(DATASETS)
fig, axes = plt.subplots(2, n, figsize=(n * 2.8, 9))
if n == 1:
    axes = axes.reshape(2, 1)

for col, d in enumerate(DATASETS):
    asp_th    = _thumb(d['paths']['anime_stitch'])
    simple_th = _thumb(d['paths']['simple_stitch'])
    verdict   = d['comparison'].get('verdict', '')
    verdict_color = {
        'asp_better':       '#4ecdc4',
        'simple_better':    '#ff6b6b',
        'comparable':       '#ffd93d',
        'insufficient_data':'#aaa',
    }.get(verdict, 'white')

    axes[0, col].imshow(cv2.cvtColor(asp_th,    cv2.COLOR_BGR2RGB), aspect='auto')
    axes[1, col].imshow(cv2.cvtColor(simple_th, cv2.COLOR_BGR2RGB), aspect='auto')

    short_name = d['name'].replace('asp_test', 'T')
    fallback_marker = ' ⚡' if d['used_fallback'] else ''
    axes[0, col].set_title(f'{short_name}{fallback_marker}',
                            fontsize=8, color=verdict_color)
    axes[0, col].axis('off')
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('ASP', fontsize=10, color='#4ecdc4', rotation=0, labelpad=40)
axes[1, 0].set_ylabel('Simple', fontsize=10, color='#ff6b6b', rotation=0, labelpad=40)

fig.suptitle('All Datasets — ASP (top) vs Simple Stitch (bottom)\n'
             'Title colour: teal=ASP better, red=Simple better, yellow=comparable  ⚡=fallback',
             fontsize=10)
plt.tight_layout()
plt.show()

## 9. Stage Intermediate Outputs Viewer

Shows the pipeline stage images saved by the benchmark for a selected dataset.

In [ ]:
STAGE_DATASET = 'asp_test1'   # ← change this

d = next((r for r in DATASETS if r['name'] == STAGE_DATASET), None)
if d is None:
    print(f'{STAGE_DATASET} not found.')
else:
    sd = d['paths']['stage_dir']
    N_fr = d['frames']['count']

    stage_groups = [
        ('Normalised Frames (Stage 2)',
         [os.path.join(sd, f'stage02_normalised_frame{i:02d}.png') for i in range(N_fr)]),
        ('Photometric Corrected Frames (Stage 3)',
         [os.path.join(sd, f'stage03_basic_corrected_frame{i:02d}.png') for i in range(N_fr)]),
        ('BG Masks (Stage 4)',
         [os.path.join(sd, f'stage04_bgmask_frame{i:02d}.png') for i in range(N_fr)]),
    ]

    for group_title, paths in stage_groups:
        existing = [p for p in paths if os.path.exists(p)]
        if not existing:
            print(f'  {group_title}: no files found')
            continue
        ncols = min(len(existing), 6)
        fig, axes = plt.subplots(1, ncols, figsize=(ncols * 3.5, 3))
        if ncols == 1:
            axes = [axes]
        for ax, fpath in zip(axes, existing[:ncols]):
            img = cv2.imread(fpath)
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), aspect='auto')
            ax.set_title(Path(fpath).name, fontsize=6)
            ax.axis('off')
        fig.suptitle(f'{STAGE_DATASET} — {group_title}', fontsize=9)
        plt.tight_layout()
        plt.show()

    # Render and composite
    for fname, label in [
        ('stage09_temporal_render.png', 'Stage 9 — Temporal Median Render'),
        ('stage11_fg_composite.png',    'Stage 11 — FG Composite'),
    ]:
        fpath = os.path.join(sd, fname)
        img   = cv2.imread(fpath)
        if img is not None:
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), aspect='auto')
            ax.set_title(f'{STAGE_DATASET} — {label}', fontsize=9)
            ax.axis('off')
            plt.tight_layout()
            plt.show()

## 10. Metrics Correlation & Heatmap

In [ ]:
# Build a metrics matrix for correlation analysis
metric_keys = [
    ('asp_sharpness',    lambda d: d['metrics_asp'].get('sharpness', 0)),
    ('asp_coverage',     lambda d: d['metrics_asp'].get('coverage', 0)),
    ('asp_ghosting',     lambda d: d['metrics_asp'].get('ghosting_score', 0)),
    ('asp_seam_grad',    lambda d: d['metrics_asp'].get('seam_gradient', 0)),
    ('asp_entropy',      lambda d: d['metrics_asp'].get('color_entropy', 0)),
    ('sim_sharpness',    lambda d: d['metrics_simple'].get('sharpness', 0)),
    ('sim_coverage',     lambda d: d['metrics_simple'].get('coverage', 0)),
    ('sim_ghosting',     lambda d: d['metrics_simple'].get('ghosting_score', 0)),
    ('ssim',             lambda d: d['comparison'].get('ssim') or 0),
    ('total_time',       lambda d: d['time'].get('total_sec', 0)),
    ('dy_cv',            lambda d: d['alignment'].get('dy_cv', 0)),
    ('n_frames',         lambda d: d['frames'].get('count', 0)),
    ('gain_range',       lambda d: (d['photometric'].get('gain_range', [1,1])[1]
                                    - d['photometric'].get('gain_range', [1,1])[0])),
]

mat = np.array([[fn(d) for fn in [v for _, v in metric_keys]] for d in DATASETS],
               dtype=np.float64)

# Pearson correlation matrix
from numpy.linalg import norm
def _pearson_matrix(M):
    n, k = M.shape
    C = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            a = M[:, i] - M[:, i].mean()
            b = M[:, j] - M[:, j].mean()
            denom = (norm(a) * norm(b))
            C[i, j] = float(a @ b / denom) if denom > 1e-10 else 0.0
    return C

corr = _pearson_matrix(mat)
labels = [k for k, _ in metric_keys]

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
for i in range(len(labels)):
    for j in range(len(labels)):
        val = corr[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6,
                color='black' if abs(val) < 0.6 else 'white')
plt.colorbar(im, ax=ax, fraction=0.03, label='Pearson r')
ax.set_title('Metric Correlation Matrix (all datasets)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Per-dataset metrics heatmap (normalised per column)
mat_norm = mat.copy()
for col in range(mat_norm.shape[1]):
    col_min, col_max = mat_norm[:, col].min(), mat_norm[:, col].max()
    if col_max > col_min:
        mat_norm[:, col] = (mat_norm[:, col] - col_min) / (col_max - col_min)

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(mat_norm.T, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(DATASETS)))
ax.set_xticklabels(short, fontsize=8)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
for i, d in enumerate(DATASETS):
    if d['used_fallback']:
        ax.axvline(i, color='yellow', linewidth=2, alpha=0.35)
plt.colorbar(im, ax=ax, fraction=0.015, label='normalised value (per metric)')
ax.set_title('Per-Dataset Metrics Heatmap  (green=high, red=low per metric; yellow=fallback)',
             fontsize=10)
plt.tight_layout()
plt.show()

## 11. Performance Insights

In [ ]:
def _insight_row(label, obj):
    if obj is None:
        return f'| {label} | — |'
    return f'| {label} | **{obj["name"]}** ({obj.get("value", "")})|'

rows = [
    '| Insight | Dataset |',
    '|---------|---------|',
    _insight_row('Slowest dataset',       INSIGHTS.get('slowest_dataset')),
    _insight_row('Fastest dataset',       INSIGHTS.get('fastest_dataset')),
    _insight_row('Best ASP sharpness',    INSIGHTS.get('best_asp_sharpness')),
    _insight_row('Worst ASP sharpness',   INSIGHTS.get('worst_asp_sharpness')),
    _insight_row('Most ASP ghosting',     INSIGHTS.get('most_asp_ghosting')),
    _insight_row('Least ASP ghosting',    INSIGHTS.get('least_asp_ghosting')),
]
display(Markdown('### Performance Insights\n' + '\n'.join(rows)))

asp_better = INSIGHTS.get('datasets_asp_better_than_simple', [])
sim_better = INSIGHTS.get('datasets_simple_better_than_asp', [])
failed     = INSIGHTS.get('datasets_alignment_failed', [])

display(Markdown(
    f'**ASP better ({len(asp_better)}):** {", ".join(asp_better) or "none"}\n\n'
    f'**Simple better ({len(sim_better)}):** {chr(10)}{chr(10).join("- " + n for n in sim_better) or "none"}\n\n'
    f'**Alignment failed ({len(failed)}):** {chr(10)}{chr(10).join("- " + n for n in failed) or "none"}'
))

In [ ]:
# Radar chart: average metric profile — ASP vs Simple
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

radar_metrics = [
    ('Sharpness',  'sharpness',     True),
    ('Coverage',   'coverage',      True),
    ('Low Ghost',  'ghosting_score', False),   # invert: lower ghosting = better
    ('Low Seam∇',  'seam_gradient',  False),   # invert
    ('Entropy',    'color_entropy',  True),
]

def _radar_vals(pipeline_key):
    vals = []
    for label, key, higher_better in radar_metrics:
        v_list = [d[pipeline_key].get(key, 0) for d in DATASETS if d[pipeline_key]]
        v_mean = float(np.mean(v_list)) if v_list else 0.0
        vals.append(v_mean)
    return vals

asp_raw = _radar_vals('metrics_asp')
sim_raw = _radar_vals('metrics_simple')

# Normalise jointly so both pipelines share the same scale
combined_max = [max(a, b, 1e-9) for a, b in zip(asp_raw, sim_raw)]
asp_norm = [a/m for a, m in zip(asp_raw, combined_max)]
sim_norm = [s/m for s, m in zip(sim_raw, combined_max)]
# Invert metrics where lower = better
for i, (_, _, higher_better) in enumerate(radar_metrics):
    if not higher_better:
        asp_norm[i] = 1.0 - asp_norm[i]
        sim_norm[i] = 1.0 - sim_norm[i]

N_axes = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N_axes, endpoint=False).tolist()
angles += angles[:1]
asp_plot  = asp_norm  + asp_norm[:1]
sim_plot  = sim_norm  + sim_norm[:1]
ax_labels = [m[0] for m in radar_metrics]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'polar': True})
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#12121f')
ax.plot(angles, asp_plot,  'o-', color='#4ecdc4', linewidth=2, label='ASP')
ax.fill(angles, asp_plot,  alpha=0.15, color='#4ecdc4')
ax.plot(angles, sim_plot,  's-', color='#ff6b6b', linewidth=2, label='Simple')
ax.fill(angles, sim_plot,  alpha=0.15, color='#ff6b6b')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(ax_labels, color='white', fontsize=9)
ax.set_yticklabels([])
ax.set_ylim(0, 1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1),
          facecolor='#2a2a3e', labelcolor='white', fontsize=9)
ax.set_title('Average Quality Radar\n(normalised, higher = better on all axes)',
             color='white', fontsize=10, pad=20)
ax.tick_params(colors='white')
ax.spines['polar'].set_color('#555')
plt.tight_layout()
plt.show()

## 12. Edge Details for a Selected Dataset

In [ ]:
EDGE_DATASET = 'asp_test1'   # ← change this

d = next((r for r in DATASETS if r['name'] == EDGE_DATASET), None)
if d is None:
    print(f'{EDGE_DATASET} not found.')
else:
    edges = d['matching'].get('edges', [])
    print(f'{EDGE_DATASET} — {len(edges)} filtered edges')
    print(f'  Raw: {d["matching"]["raw_edges"]}  Filtered: {d["matching"]["filtered_edges"]}')
    print(f'  Methods: {d["matching"]["methods"]}')
    print()
    if edges:
        print(f'  {"i":>3} {"j":>3} {"method":>12} {"n_pts":>6} {"weight":>8} {"tx":>8} {"ty":>8}')
        print('  ' + '─' * 56)
        for e in edges:
            print(f'  {e["i"]:>3} {e["j"]:>3} {e["method"]:>12} '
                  f'{e["n_pts"]:>6} {e["weight"]:>8.4f} {e["tx"]:>8.2f} {e["ty"]:>8.2f}')

        # Plot edge weights and shifts
        if len(edges) > 1:
            fig, axes = plt.subplots(1, 3, figsize=(13, 4))
            edge_ids = [f'{e["i"]}-{e["j"]}' for e in edges]
            edge_w   = [e['weight'] for e in edges]
            edge_tx  = [e['tx'] for e in edges]
            edge_ty  = [e['ty'] for e in edges]
            xi = np.arange(len(edges))

            axes[0].bar(xi, edge_w, color='#4ecdc4', alpha=0.85)
            axes[0].set_xticks(xi)
            axes[0].set_xticklabels(edge_ids, fontsize=8)
            axes[0].set_title('Edge Confidence Weights')
            axes[0].set_ylabel('Weight')
            axes[0].grid(axis='y', alpha=0.2)

            axes[1].bar(xi, edge_tx, color='#ffd93d', alpha=0.85)
            axes[1].set_xticks(xi)
            axes[1].set_xticklabels(edge_ids, fontsize=8)
            axes[1].set_title('tx per Edge')
            axes[1].set_ylabel('px')
            axes[1].grid(axis='y', alpha=0.2)

            axes[2].bar(xi, edge_ty, color='#fd79a8', alpha=0.85)
            axes[2].set_xticks(xi)
            axes[2].set_xticklabels(edge_ids, fontsize=8)
            axes[2].set_title('ty per Edge')
            axes[2].set_ylabel('px')
            axes[2].grid(axis='y', alpha=0.2)

            plt.suptitle(f'{EDGE_DATASET} — Edge Statistics', fontsize=10)
            plt.tight_layout()
            plt.show()
    else:
        print('  No edge data available (fallback or old JSON format).')

## 13. Raw JSON Explorer

In [ ]:
# Print the full JSON for any one dataset
RAW_DATASET = 'asp_test1'   # ← change this

d = next((r for r in DATASETS if r['name'] == RAW_DATASET), None)
if d:
    # Suppress the large 'edges' array to keep output readable
    d_display = {k: v for k, v in d.items() if k != 'matching'}
    d_display['matching'] = {
        k: v for k, v in d['matching'].items() if k != 'edges'
    }
    d_display['matching']['edges'] = f'[{len(d["matching"].get("edges", []))} edges — omitted]'
    print(json.dumps(d_display, indent=2))